# VAE diagnostics

A compact view of one trained VAE: prior samples, reconstructions, class prototypes, latent-dimension activity, and hole scores.

In [1]:
from pathlib import Path
import math
import re
import sys

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset


REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

from domain_knowledge_analysis import utils
from domain_knowledge_analysis.math.gaussian import gaussian_log_prob
from domain_knowledge_analysis.scoring.signals import HoleScoreEstimator

## Settings

In [2]:
CHECKPOINT = REPO_ROOT / "runs/vae_organamnist_lr_0.001_07-07_15-50-03/checkpoints/last.pt"
BATCH_SIZE = 256
HOLE_SCORE_SAMPLES = 1000
LEAKAGE_FREE_COMPARISON = True
SEED = 42

## Analysis

In [3]:
class VaeDiagnostics:
    def __init__(self, checkpoint_path, batch_size=256, hole_score_samples=1000, seed=42, leakage_free=True):
        self.checkpoint_path = Path(checkpoint_path)
        self.batch_size = batch_size
        self.hole_score_samples = hole_score_samples
        self.seed = seed
        self.leakage_free = leakage_free
        self.device = utils.get_device()

        checkpoint = torch.load(self.checkpoint_path, map_location="cpu", weights_only=False)
        self.config = checkpoint["config"]
        self.model = utils.create_model(self.config)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model = self.model.to(self.device).eval()

        run_name = self.checkpoint_path.parents[1].name
        suffix = re.search(r"\d{4}_(.+)$", run_name)
        self.name = suffix.group(1) if suffix else run_name
        self.latent_dim = self.config["model"]["encoder"]["latent_dim"]

        train_loader, validation_loader = utils.create_training_dataloaders(self.config)
        test_dataset = utils.create_dataset(self.config, self.config["dataset"]["name"], train=False)

        self.train_loader = self._loader(train_loader.dataset)
        self.validation_loader = self._loader(validation_loader.dataset)
        self.test_loader = self._loader(test_dataset)

        self.train_mean, self.train_log_variance, self.train_labels = self._encode(self.train_loader)

        self.hole_estimator = HoleScoreEstimator(self.train_loader, self.model, self.config["model"]["name"].lower())
        self.hole_estimator.q_in_mean = self.train_mean.to(self.device)
        self.hole_estimator.q_in_log_variance = self.train_log_variance.to(self.device)

        print(f"{self.name} | {self.config['dataset']['name']} | {self.model.decoder_distribution_name} | {self.latent_dim} latent dimensions | {self.device}")

    def _loader(self, dataset):
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)

    @torch.no_grad()
    def _encode(self, loader):
        means = []
        log_variances = []
        labels = []

        for images, batch_labels in loader:
            mean, log_variance = self.model.encoder(images.to(self.device))
            means.append(mean.cpu())
            log_variances.append(log_variance.cpu())
            labels.append(batch_labels.cpu().flatten())

        return torch.cat(means), torch.cat(log_variances), torch.cat(labels)

    def _images(self, axes, images, titles=None):
        for index, (axis, image) in enumerate(zip(axes.flat, images)):
            axis.imshow(image.squeeze().cpu(), cmap="gray", vmin=0, vmax=1)
            axis.axis("off")

            if titles is not None:
                axis.set_title(titles[index])

    @torch.no_grad()
    def prior_samples(self):
        torch.manual_seed(self.seed)
        images = self.model.generate_images(100).cpu()
        figure, axes = plt.subplots(10, 10, figsize=(10, 10))
        self._images(axes, images)
        figure.suptitle(f"{self.name}: samples from the prior")
        plt.tight_layout()
        plt.show()

    @torch.no_grad()
    def reconstructions(self):
        images = torch.stack([self.test_loader.dataset[index][0] for index in range(10)])
        reconstructions = self.model.reconstruct_images(images).cpu()
        figure, axes = plt.subplots(2, 10, figsize=(15, 3))
        self._images(axes[0], images)
        self._images(axes[1], reconstructions)
        axes[0, 0].set_ylabel("Original")
        axes[1, 0].set_ylabel("Reconstruction")
        figure.suptitle(f"{self.name}: test reconstructions")
        plt.tight_layout()
        plt.show()

    @torch.no_grad()
    def class_prototypes(self):
        class_ids = self.train_labels.unique(sorted=True)
        latents = torch.stack([self.train_mean[self.train_labels == class_id].mean(dim=0) for class_id in class_ids])
        images = self.model.generate_images(len(latents), latents).cpu()
        titles = [str(class_id.item()) for class_id in class_ids]
        figure, axes = plt.subplots(1, len(images), figsize=(15, 2))
        self._images(axes, images, titles)
        figure.suptitle(f"{self.name}: decoded class prototypes")
        plt.tight_layout()
        plt.show()

    def latent_statistics(self):
        active_mean_variance_threshold = 1e-2
        active_kl_threshold = 1e-2

        means = self.train_mean.float()
        log_variances = self.train_log_variance.float()
        posterior_variances = log_variances.exp()

        mean_of_posterior_means = means.mean(dim=0)
        variance_of_posterior_means = means.var(dim=0, unbiased=False)
        mean_posterior_variance = posterior_variances.mean(dim=0)
        aggregate_posterior_variance = variance_of_posterior_means + mean_posterior_variance
        mean_kl_per_dimension = 0.5 * (posterior_variances + means.square() - 1 - log_variances).mean(dim=0)

        active_by_mean_variance = variance_of_posterior_means > active_mean_variance_threshold
        active_by_kl = mean_kl_per_dimension > active_kl_threshold
        pruned = ~(active_by_mean_variance | active_by_kl)

        print(f"{self.name} — {self.model.decoder_distribution_name}")
        print("dim | E[mu]    | Var_x(mu) | E[sigma^2] | Var_q(z) | mean KL  | pruned")
        print("----|----------|-----------|------------|----------|----------|-------")

        for dimension in range(self.latent_dim):
            print(f"{dimension:>3} | {mean_of_posterior_means[dimension].item():>8.4f} | {variance_of_posterior_means[dimension].item():>9.5f} | {mean_posterior_variance[dimension].item():>10.5f} | {aggregate_posterior_variance[dimension].item():>8.5f} | {mean_kl_per_dimension[dimension].item():>8.5f} | {str(bool(pruned[dimension])):>6}")

        pruned_dimensions = torch.where(pruned)[0].tolist()
        print(f"\nPruned dimensions: {len(pruned_dimensions)}/{self.latent_dim} {pruned_dimensions}")

        dimensions = torch.arange(self.latent_dim)
        figure, axes = plt.subplots(2, 2, figsize=(12, 7))

        axes[0, 0].bar(dimensions, mean_of_posterior_means)
        axes[0, 0].set_title("Mean posterior mean")

        axes[0, 1].bar(dimensions, variance_of_posterior_means)
        axes[0, 1].axhline(active_mean_variance_threshold, color="black", linestyle="--")
        axes[0, 1].set_title("Posterior-mean activity")

        axes[1, 0].bar(dimensions, mean_kl_per_dimension)
        axes[1, 0].axhline(active_kl_threshold, color="black", linestyle="--")
        axes[1, 0].set_title("Mean KL")

        axes[1, 1].bar(dimensions, aggregate_posterior_variance, label="aggregate")
        axes[1, 1].bar(dimensions, mean_posterior_variance, label="conditional")
        axes[1, 1].set_title("Posterior variances")
        axes[1, 1].legend()

        for axis in axes.flat:
            axis.set_xlabel("Latent dimension")

        figure.suptitle(f"{self.name}: latent-dimension summary")
        plt.tight_layout()
        plt.show()

    def _query_loader(self, loader, seed):
        count = min(self.hole_score_samples, len(loader.dataset))
        generator = torch.Generator().manual_seed(seed)
        indices = torch.randperm(len(loader.dataset), generator=generator)[:count]
        dataset = Subset(loader.dataset, indices.tolist())
        return self._loader(dataset)

    @torch.no_grad()
    def _scores(self, loader):
        scores = []

        for images, _ in loader:
            mean, log_variance = self.model.encoder(images.to(self.device))
            latents = self.model.reparametrize(mean, log_variance)
            scores.append(self.hole_estimator.estimate_h(latents).cpu())

        return torch.cat(scores)

    def _print_distribution_summary(self, distributions):
        print(f"{'distribution':<30} {'n':>6} {'mean':>12} {'std':>12}")

        for name, values in distributions.items():
            values = torch.as_tensor(values).float()
            print(f"{name:<30} {len(values):>6} {values.mean().item():>12.6f} {values.std(unbiased=True).item():>12.6f}")

    def _plot_distributions(self, distributions, title):
        for name, values in distributions.items():
            plt.hist(values.numpy(), bins=50, density=True, alpha=0.45, label=name)

        plt.xlabel("H(z)")
        plt.ylabel("Density")
        plt.title(title)
        plt.legend()
        plt.tight_layout()
        plt.show()

    def hole_scores(self):
        torch.manual_seed(self.seed)

        scores = {
            "training": self._scores(self._query_loader(self.train_loader, self.seed)),
            "validation": self._scores(self._query_loader(self.validation_loader, self.seed + 1)),
            "test": self._scores(self._query_loader(self.test_loader, self.seed + 2)),
        }

        self._print_distribution_summary(scores)
        self._plot_distributions(scores, f"{self.name}: hole scores using the training aggregate posterior")

    @torch.no_grad()
    def _log_q_leave_one_out(self, latents, component_indices, query_batch_size=64):
        means = self.hole_estimator.q_in_mean
        log_variances = self.hole_estimator.q_in_log_variance

        if len(latents) != len(component_indices):
            raise ValueError("Each latent must identify its generating mixture component.")

        if len(means) < 2:
            raise ValueError("Leave-one-out requires at least two mixture components.")

        inverse_variances = torch.exp(-log_variances)
        log_q_batches = []

        for start in range(0, len(latents), query_batch_size):
            stop = min(start + query_batch_size, len(latents))
            z_batch = latents[start:stop].unsqueeze(1)
            component_log_q = -0.5 * torch.sum(math.log(2.0 * math.pi) + log_variances.unsqueeze(0) + (z_batch - means.unsqueeze(0)).square() * inverse_variances.unsqueeze(0), dim=-1)
            row_indices = torch.arange(stop - start, device=self.device)
            generating_components = component_indices[start:stop].to(self.device)
            component_log_q[row_indices, generating_components] = -torch.inf
            log_q_batch = torch.logsumexp(component_log_q, dim=1) - math.log(len(means) - 1)
            log_q_batches.append(log_q_batch)

        return torch.cat(log_q_batches)

    @torch.no_grad()
    def leakage_comparison(self):
        if not self.leakage_free:
            return

        count = min(self.hole_score_samples, len(self.train_mean))
        index_generator = torch.Generator().manual_seed(self.seed)
        component_indices = torch.randperm(len(self.train_mean), generator=index_generator)[:count]
        means = self.train_mean[component_indices].to(self.device)
        log_variances = self.train_log_variance[component_indices].to(self.device)

        latent_generator = torch.Generator(device="cpu").manual_seed(self.seed + 200)
        noise = torch.randn(means.shape, generator=latent_generator, dtype=means.dtype, device="cpu").to(self.device)
        latents = means + noise * torch.exp(0.5 * log_variances)

        self_included = self.hole_estimator.estimate_h(latents).detach().cpu()
        log_q_leave_one_out = self._log_q_leave_one_out(latents, component_indices, query_batch_size=64)
        log_prior = gaussian_log_prob(latents, torch.zeros_like(latents), torch.zeros_like(latents))
        leave_one_out = (log_prior - log_q_leave_one_out).detach().cpu()

        torch.manual_seed(self.seed + 201)
        held_out = self._scores(self._query_loader(self.validation_loader, self.seed + 1)).detach().cpu()

        distributions = {"self included": self_included, "leave one out": leave_one_out, "held-out validation": held_out}
        self._print_distribution_summary(distributions)
        self._plot_distributions(distributions, f"{self.name}: calibration leakage comparison")

In [4]:
diagnostics = VaeDiagnostics(
    CHECKPOINT,
    batch_size=BATCH_SIZE,
    hole_score_samples=HOLE_SCORE_SAMPLES,
    seed=SEED,
    leakage_free=LEAKAGE_FREE_COMPARISON,
)

KeyError: 'loss'

## Prior samples

In [ ]:
diagnostics.prior_samples()

## Test reconstructions

In [ ]:
diagnostics.reconstructions()

## Decoded class prototypes

In [ ]:
diagnostics.class_prototypes()

## Latent dimensions

In [ ]:
diagnostics.latent_statistics()

## Hole scores

In [ ]:
diagnostics.hole_scores()

## Calibration leakage

In [ ]:
diagnostics.leakage_comparison()